<a href="https://colab.research.google.com/github/wan-chtvlv/spatial-coastal-ses/blob/main/A1_1_tutorial_bulk_geocoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A1-1 Tutorial from SpatialThoughts
GeoPandas - Bulk Geocoding Addresses
https://www.geopythontutorials.com/notebooks/geopandas_bulk_geocoding.html

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
  !pip install leafmap mapclassify

In [2]:
import os
import re
import pandas as pd
import geopandas as gpd
import leafmap.foliumap as leafmap
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from zipfile import ZipFile

In [7]:
data_folder = './A1-1-data/'
output_folder = './A1-1-output/'

if not os.path.exists(data_folder):
    os.makedirs(data_folder)
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def download(url):
    filename = os.path.join(data_folder, os.path.basename(url))
    if not os.path.exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded' + local)

data_url = 'https://github.com/spatialthoughts/geopython-tutorials/releases/download/data/'
download(data_url + 'hurricane_evacuation_centers.xlsx')

In [10]:
data = 'hurricane_evacuation_centers.xlsx'
data_path = os.path.join(data_folder, data)
address_df = pd.read_excel(data_path, skiprows=[0])
address_df

,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE
0,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y
1,Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y
2,Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y
3,Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y
4,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y
...,...,...,...,...,...,...,...
59,Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y
60,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y
61,Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y
62,Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y


Making street numbers (e.g. 3, 4) into ordinal number (3rd, 4th), so that it works with Nominatim geocoder format.

In [14]:
def make_ordinal(match):
  # match.group is a function to call on parentheses pair:
  # group(0) is always the whole match, while group(1) is the first group on the string, and so on.
  # the groups are defined in def update_address(row)
  n = int(match.group(1))
  # If the number is between 11-13, then it ends with th
  if 11 <= (n%100) <= 13:
    suffix = 'th'
  else:
    suffix = ['th', 'st', 'nd', 'rd', 'th'][min(n%10, 4)]
  return str(n) + suffix + match.group(2)

def update_address(row):
  old_address = row['ADDRESS']
  # group pattern is defined here. It only looks for the pattern where there are (\d+) decimal digits, a (\s+) whitespace, and the string of street type identifier.
  # More resources: https://docs.python.org/3/howto/regex.html
  # The objective of this is to identify the street names that are numbered and make them ordinal.
  pattern = r'(\d+)(\s+(?:Street|Avenue|Blvd|Drive))'
  result = re.sub(pattern, make_ordinal, old_address)
  return result

address_df['ADDRESS_FIXED'] = address_df.apply(update_address, axis=1)
address_df

,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE,ADDRESS_FIXED
0,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y,147-26 25th Drive
1,Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y,141 Macon Street
2,Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y,544 7th Avenue
3,Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y,2780 Reservoir Avenue
4,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y,7002 4th Avenue
...,...,...,...,...,...,...,...,...
59,Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y,730 Bryant Avenue
60,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y,2760 Briggs Avenue
61,Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y,45-30 36th Street
62,Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y,1827 Archer Street


In [16]:
# Knowing that all of these addresses are in NYC, we can also have a full address column added here.
address_df['Full_Address'] = (
    address_df['ADDRESS_FIXED'] + ',' +
    'NYC' + ',' +
    address_df['STATE']+ ',' +
    address_df['ZIP_CODE'].astype(str))
address_df

,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE,ADDRESS_FIXED,Full_Address
0,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y,147-26 25th Drive,"147-26 25th Drive,NYC,NY,11354"
1,Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y,141 Macon Street,"141 Macon Street,NYC,NY,11216"
2,Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y,544 7th Avenue,"544 7th Avenue,NYC,NY,11215"
3,Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y,2780 Reservoir Avenue,"2780 Reservoir Avenue,NYC,NY,10468"
4,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y,7002 4th Avenue,"7002 4th Avenue,NYC,NY,11209"
...,...,...,...,...,...,...,...,...,...
59,Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y,730 Bryant Avenue,"730 Bryant Avenue,NYC,NY,10474"
60,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y,2760 Briggs Avenue,"2760 Briggs Avenue,NYC,NY,10458"
61,Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y,45-30 36th Street,"45-30 36th Street,NYC,NY,11101"
62,Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y,1827 Archer Street,"1827 Archer Street,NYC,NY,10460"


Geocoding Method 1: Pandas

In [17]:
# https://tqdm.github.io/
# tqdm library shows a progress meter while we run loops.
from tqdm.notebook import tqdm
tqdm.pandas()
# [Nominatim] is a function in the [geopy.geocoders] library imported at the beginning of this notebook.https://nominatim.org/
# The output of this function is an object containing the address along with coordinates, which needs to be extracted.
locator = Nominatim(user_agent='wan', timeout=10)
# [RateLimiter] is a function in [geopy.extra.rate_limiter] library imported at the beginning of this notebook.
geocode = RateLimiter(locator.geocode, min_delay_seconds=1)

address_df_pd = address_df.copy()
address_df_pd['location'] = address_df_pd['Full_Address'].progress_apply(geocode)
address_df_pd

  0%|          | 0/64 [00:00<?, ?it/s]

,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE,ADDRESS_FIXED,Full_Address,location
0,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y,147-26 25th Drive,"147-26 25th Drive,NYC,NY,11354",(J.H.S. 185 - Edward Bleeker Junior High Schoo...
1,Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y,141 Macon Street,"141 Macon Street,NYC,NY,11216","(141, Macon Street, Bedford-Stuyvesant, Brookl..."
2,Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y,544 7th Avenue,"544 7th Avenue,NYC,NY,11215","(Middle School 88, 544, 7th Avenue, Greenwood ..."
3,Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y,2780 Reservoir Avenue,"2780 Reservoir Avenue,NYC,NY,10468","(Celia Cruz Bronx High School of Music, 2780, ..."
4,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y,7002 4th Avenue,"7002 4th Avenue,NYC,NY,11209",(P.S. / I.S. 30 - The Mary White Ovington Scho...
...,...,...,...,...,...,...,...,...,...,...
59,Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y,730 Bryant Avenue,"730 Bryant Avenue,NYC,NY,10474","(Bronx Academy for Multi-Media, 730, Bryant Av..."
60,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y,2760 Briggs Avenue,"2760 Briggs Avenue,NYC,NY,10458",(P.S. 46 - The Edgar Allan Poe Literacy Develo...
61,Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y,45-30 36th Street,"45-30 36th Street,NYC,NY,11101","(Aviation High School, 45-30, 36th Street, Sun..."
62,Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y,1827 Archer Street,"1827 Archer Street,NYC,NY,10460","(1827, Archer Street, Parkchester, The Bronx, ..."


In [18]:
# Extracting lat lon from the address location object created from the Nominatim function above.
address_df_pd['latitude'] = address_df_pd['location'].apply(lambda loc: loc.latitude if loc else None)
address_df_pd['longitude'] = address_df_pd['location'].apply(lambda loc: loc.longitude if loc else None)
address_df_pd

,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE,ADDRESS_FIXED,Full_Address,location,latitude,longitude
0,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y,147-26 25th Drive,"147-26 25th Drive,NYC,NY,11354",(J.H.S. 185 - Edward Bleeker Junior High Schoo...,40.774912,-73.818619
1,Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y,141 Macon Street,"141 Macon Street,NYC,NY,11216","(141, Macon Street, Bedford-Stuyvesant, Brookl...",40.681918,-73.945597
2,Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y,544 7th Avenue,"544 7th Avenue,NYC,NY,11215","(Middle School 88, 544, 7th Avenue, Greenwood ...",40.660492,-73.988570
3,Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y,2780 Reservoir Avenue,"2780 Reservoir Avenue,NYC,NY,10468","(Celia Cruz Bronx High School of Music, 2780, ...",40.870595,-73.897498
4,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y,7002 4th Avenue,"7002 4th Avenue,NYC,NY,11209",(P.S. / I.S. 30 - The Mary White Ovington Scho...,40.633495,-74.024544
...,...,...,...,...,...,...,...,...,...,...,...,...
59,Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y,730 Bryant Avenue,"730 Bryant Avenue,NYC,NY,10474","(Bronx Academy for Multi-Media, 730, Bryant Av...",40.815693,-73.885498
60,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y,2760 Briggs Avenue,"2760 Briggs Avenue,NYC,NY,10458",(P.S. 46 - The Edgar Allan Poe Literacy Develo...,40.867194,-73.890166
61,Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y,45-30 36th Street,"45-30 36th Street,NYC,NY,11101","(Aviation High School, 45-30, 36th Street, Sun...",40.743314,-73.929587
62,Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y,1827 Archer Street,"1827 Archer Street,NYC,NY,10460","(1827, Archer Street, Parkchester, The Bronx, ...",40.838106,-73.865790


The tutorial also shows how to review and manually manipulate data entries that failed to geocode. However, as it's not what I aim to do and much of the data is not applied in developing countries, I decided to skip the section.

In [20]:
# Make geometries from the coordinates with GeoPandas (gpd) function [points_from_xy]
geometry = gpd.points_from_xy(address_df_pd.longitude, address_df_pd.latitude)
address_gdf = gpd.GeoDataFrame(address_df_pd, crs='EPSG:4326', geometry=geometry)
address_gdf

,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE,ADDRESS_FIXED,Full_Address,location,latitude,longitude,geometry
0,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y,147-26 25th Drive,"147-26 25th Drive,NYC,NY,11354",(J.H.S. 185 - Edward Bleeker Junior High Schoo...,40.774912,-73.818619,POINT (-73.81862 40.77491)
1,Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y,141 Macon Street,"141 Macon Street,NYC,NY,11216","(141, Macon Street, Bedford-Stuyvesant, Brookl...",40.681918,-73.945597,POINT (-73.9456 40.68192)
2,Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y,544 7th Avenue,"544 7th Avenue,NYC,NY,11215","(Middle School 88, 544, 7th Avenue, Greenwood ...",40.660492,-73.988570,POINT (-73.98857 40.66049)
3,Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y,2780 Reservoir Avenue,"2780 Reservoir Avenue,NYC,NY,10468","(Celia Cruz Bronx High School of Music, 2780, ...",40.870595,-73.897498,POINT (-73.8975 40.87059)
4,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y,7002 4th Avenue,"7002 4th Avenue,NYC,NY,11209",(P.S. / I.S. 30 - The Mary White Ovington Scho...,40.633495,-74.024544,POINT (-74.02454 40.63349)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y,730 Bryant Avenue,"730 Bryant Avenue,NYC,NY,10474","(Bronx Academy for Multi-Media, 730, Bryant Av...",40.815693,-73.885498,POINT (-73.8855 40.81569)
60,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y,2760 Briggs Avenue,"2760 Briggs Avenue,NYC,NY,10458",(P.S. 46 - The Edgar Allan Poe Literacy Develo...,40.867194,-73.890166,POINT (-73.89017 40.86719)
61,Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y,45-30 36th Street,"45-30 36th Street,NYC,NY,11101","(Aviation High School, 45-30, 36th Street, Sun...",40.743314,-73.929587,POINT (-73.92959 40.74331)
62,Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y,1827 Archer Street,"1827 Archer Street,NYC,NY,10460","(1827, Archer Street, Parkchester, The Bronx, ...",40.838106,-73.865790,POINT (-73.86579 40.83811)


Geocoding Method 2: GeoPandas

In [23]:
geocoder_options = {
     'user_agent': 'wan',
     'timeout': 10,
}

address_df_gpd = address_df.copy()

# GeoPandas uses the value of 1-second for rate limiting Nominatim queries
gdf = gpd.tools.geocode(
    address_df_gpd['ADDRESS_FIXED'],
    provider='Nominatim',
    **geocoder_options)

gdf

,geometry,address
0,POINT (-73.81862 40.77491),J.H.S. 185 - Edward Bleeker Junior High School...
1,POINT (-73.9456 40.68192),"141, Macon Street, Bedford-Stuyvesant, Brookly..."
2,POINT (-73.98857 40.66049),"Middle School 88, 544, 7th Avenue, Greenwood H..."
3,POINT (-73.8975 40.87059),"Celia Cruz Bronx High School of Music, 2780, R..."
4,POINT (-74.02454 40.63349),P.S. / I.S. 30 - The Mary White Ovington Schoo...
...,...,...
59,POINT (-73.8855 40.81569),"Bronx Academy for Multi-Media, 730, Bryant Ave..."
60,POINT (-73.89017 40.86719),P.S. 46 - The Edgar Allan Poe Literacy Develop...
61,POINT (-73.92959 40.74331),"Aviation High School, 45-30, 36th Street, Sunn..."
62,POINT (-73.86579 40.83811),"1827, Archer Street, The Bronx, Bronx County, ..."


In [24]:
merged = gdf.join(address_df_gpd)
merged

,geometry,address,CITY,EC_Name,ADDRESS,ZIP_CODE,BOROCODE,STATE,ACCESSIBLE,ADDRESS_FIXED,Full_Address
0,POINT (-73.81862 40.77491),J.H.S. 185 - Edward Bleeker Junior High School...,Flushing,J.H.S. 185 - Queens,147-26 25 Drive,11354,4,NY,Y,147-26 25th Drive,"147-26 25th Drive,NYC,NY,11354"
1,POINT (-73.9456 40.68192),"141, Macon Street, Bedford-Stuyvesant, Brookly...",Brooklyn,I.S. 258 - Brooklyn,141 Macon Street,11216,3,NY,Y,141 Macon Street,"141 Macon Street,NYC,NY,11216"
2,POINT (-73.98857 40.66049),"Middle School 88, 544, 7th Avenue, Greenwood H...",Brooklyn,I.S. 88 - Brooklyn,544 7 Avenue,11215,3,NY,Y,544 7th Avenue,"544 7th Avenue,NYC,NY,11215"
3,POINT (-73.8975 40.87059),"Celia Cruz Bronx High School of Music, 2780, R...",Bronx,Walton HS - X,2780 Reservoir Avenue,10468,2,NY,Y,2780 Reservoir Avenue,"2780 Reservoir Avenue,NYC,NY,10468"
4,POINT (-74.02454 40.63349),P.S. / I.S. 30 - The Mary White Ovington Schoo...,Brooklyn,P.S./I.S.30 Mary White Ovington - Brooklyn,7002 4 Avenue,11209,3,NY,Y,7002 4th Avenue,"7002 4th Avenue,NYC,NY,11209"
...,...,...,...,...,...,...,...,...,...,...,...
59,POINT (-73.8855 40.81569),"Bronx Academy for Multi-Media, 730, Bryant Ave...",Bronx,I.S. 201 - Bronx,730 Bryant Avenue,10474,2,NY,Y,730 Bryant Avenue,"730 Bryant Avenue,NYC,NY,10474"
60,POINT (-73.89017 40.86719),P.S. 46 - The Edgar Allan Poe Literacy Develop...,Bronx,P.S. 46 - Bronx,2760 Briggs Avenue,10458,2,NY,Y,2760 Briggs Avenue,"2760 Briggs Avenue,NYC,NY,10458"
61,POINT (-73.92959 40.74331),"Aviation High School, 45-30, 36th Street, Sunn...",Long Island City,Aviation HS - Q,45-30 36 Street,11101,4,NY,Y,45-30 36th Street,"45-30 36th Street,NYC,NY,11101"
62,POINT (-73.86579 40.83811),"1827, Archer Street, The Bronx, Bronx County, ...",Bronx,P.S. 102 - Bronx,1827 Archer Street,10460,2,NY,Y,1827 Archer Street,"1827 Archer Street,NYC,NY,10460"


In [25]:
merged = merged[['EC_Name', 'Full_Address', 'geometry']]
merged.rename(columns = {'EC_Name': 'Name', 'Full_Address': 'Address'}, inplace=True)
merged

,Name,Address,geometry
0,J.H.S. 185 - Queens,"147-26 25th Drive,NYC,NY,11354",POINT (-73.81862 40.77491)
1,I.S. 258 - Brooklyn,"141 Macon Street,NYC,NY,11216",POINT (-73.9456 40.68192)
2,I.S. 88 - Brooklyn,"544 7th Avenue,NYC,NY,11215",POINT (-73.98857 40.66049)
3,Walton HS - X,"2780 Reservoir Avenue,NYC,NY,10468",POINT (-73.8975 40.87059)
4,P.S./I.S.30 Mary White Ovington - Brooklyn,"7002 4th Avenue,NYC,NY,11209",POINT (-74.02454 40.63349)
...,...,...,...
59,I.S. 201 - Bronx,"730 Bryant Avenue,NYC,NY,10474",POINT (-73.8855 40.81569)
60,P.S. 46 - Bronx,"2760 Briggs Avenue,NYC,NY,10458",POINT (-73.89017 40.86719)
61,Aviation HS - Q,"45-30 36th Street,NYC,NY,11101",POINT (-73.92959 40.74331)
62,P.S. 102 - Bronx,"1827 Archer Street,NYC,NY,10460",POINT (-73.86579 40.83811)


# Visualization

In [21]:
import folium
m = leafmap.Map(width=800, height=500)
address_gdf.explore(
    m=m,
    marker_type='marker',
    marker_kwds={
        'icon': folium.Icon(color='#fdbb84', icon='hurricane', prefix='fa')
    }
)
m.zoom_to_gdf(address_gdf)
m
# OSM access blocked 403 due to the application overwhelming the servers. https://wiki.openstreetmap.org/wiki/Blocked_tiles
### I'll have to work around this in the future.

# Saving results as shapefiles

In [26]:
output_file = 'hurricane_evacuation_centers.shp'
output_path = os.path.join(output_folder, output_file)

address_gdf.to_file(filename=output_path)

In [27]:
# Zipping the shapefile for easier downloads:
output_zip_file = 'hurricane_evacuation_centers.zip'
output_zip_path = os.path.join(output_folder, output_zip_file)

sidecar_files = [
    os.path.join(output_folder, file)
    for file in os.listdir(output_folder)
    if file.endswith(('shp', 'shx', 'dbf', 'prj'))
]

with ZipFile(output_zip_path, 'w') as zip_object:
    for sidecar in sidecar_files:
      zip_object.write(sidecar, os.path.basename(sidecar))